# 18.3 Pagination, Rate Limits and Retries

**Prerequisites:** 18.1 REST and JSON, 4.3 Generators, 15.1 Why Test  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Three pagination styles — page number, **cursor**, and the `Link` header
- 🔴 Paginating with a **generator**, so the caller never sees pages (**4.3**)
- 🔴 The infinite-pagination bug, and the guard that prevents it
- Rate limits: `429`, `Retry-After` and `X-RateLimit-*`
- **Backoff with jitter** — and the thundering herd it prevents
- 🔴 What to retry and what never to retry
- `urllib3.Retry` on an `HTTPAdapter` — retries you get for free
- **Idempotency keys**, which make a `POST` safe to repeat

---

## Three problems that only appear at scale

Against a toy request, `requests.get(url).json()` is fine. Against a real API you hit three
things immediately:

1. **The collection does not fit in one response.** It is paginated.
2. **You are allowed only so many requests.** Exceed it and you get `429`.
3. **Sometimes it just fails.** A `503`, a dropped connection, a timeout.

Each has a standard answer, and each has a classic way of getting it wrong.

In [ ]:
# ---- A fake API that paginates, rate-limits and fails ----
import http.server
import json
import socket
import threading
import time
import urllib.parse

JOBS = [{"id": f"build-{i:03d}", "state": ["queued", "running", "done"][i % 3]}
        for i in range(47)]

STATE = {"limited_calls": 0, "flaky_calls": 0, "created": {}}
REQUEST_LOG = []


class BusyAPI(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *args):
        """Silence the default logging."""

    def _send(self, status, payload=None, headers=None):
        extra = dict(headers or {})
        body = b"" if payload is None else json.dumps(payload).encode("utf-8")
        self.send_response(status)
        if payload is not None:
            self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        for key, value in extra.items():
            self.send_header(key, value)
        self.end_headers()
        if body:
            self.wfile.write(body)

    def do_GET(self):
        parsed = urllib.parse.urlparse(self.path)
        query = urllib.parse.parse_qs(parsed.query)
        REQUEST_LOG.append(("GET", self.path))

        # --- page-number pagination, with a Link header ---
        if parsed.path == "/jobs":
            page = int(query.get("page", ["1"])[0])
            size = int(query.get("size", ["10"])[0])
            start = (page - 1) * size
            chunk = JOBS[start:start + size]
            headers = {}
            if start + size < len(JOBS):
                headers["Link"] = f'<{parsed.path}?page={page + 1}&size={size}>; rel="next"'
            return self._send(200, {"page": page, "size": size,
                                    "total": len(JOBS), "items": chunk}, headers)

        # --- cursor pagination ---
        if parsed.path == "/jobs/cursor":
            cursor = int(query.get("cursor", ["0"])[0])
            size = 10
            chunk = JOBS[cursor:cursor + size]
            nxt = cursor + size if cursor + size < len(JOBS) else None
            return self._send(200, {"items": chunk, "next_cursor": nxt})

        # --- a broken paginator: never signals the end ---
        if parsed.path == "/jobs/broken":
            page = int(query.get("page", ["1"])[0])
            start = (page - 1) * 10
            return self._send(200, {"items": JOBS[start:start + 10],
                                    "next_page": page + 1})     # 🔴 always set

        # --- rate limited: 429 twice, then success ---
        if parsed.path == "/limited":
            STATE["limited_calls"] += 1
            if STATE["limited_calls"] <= 2:
                return self._send(429, {"error": "rate limited"},
                                  {"Retry-After": "0",
                                   "X-RateLimit-Limit": "2",
                                   "X-RateLimit-Remaining": "0"})
            return self._send(200, {"ok": True, "call": STATE["limited_calls"]},
                              {"X-RateLimit-Limit": "2",
                               "X-RateLimit-Remaining": "1"})

        # --- flaky: 503 twice out of every three calls ---
        if parsed.path == "/flaky":
            STATE["flaky_calls"] += 1
            if STATE["flaky_calls"] % 3 != 0:
                return self._send(503, {"error": "service unavailable"})
            return self._send(200, {"ok": True, "attempts": STATE["flaky_calls"]})

        if parsed.path == "/forbidden":
            return self._send(403, {"error": "insufficient scope"})

        return self._send(404, {"error": "not found"})

    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        payload = json.loads(self.rfile.read(length) or b"{}")
        key = self.headers.get("Idempotency-Key")
        REQUEST_LOG.append(("POST", self.path))

        if key and key in STATE["created"]:          # already done this one
            return self._send(200, STATE["created"][key], {"X-Replayed": "true"})

        STATE["next_id"] = STATE.get("next_id", 900) + 1
        record = {"id": f"build-{STATE['next_id']}", **payload}
        if key:
            STATE["created"][key] = record
        return self._send(201, record)


class QuietServer(http.server.ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, *args):
        """A client hanging up is normal."""


def start_api():
    probe = socket.socket()
    probe.bind(("127.0.0.1", 0))
    port = probe.getsockname()[1]
    probe.close()
    server = QuietServer(("127.0.0.1", port), BusyAPI)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{port}"


SERVER, BASE = start_api()
print("fake API on", BASE, f"| {len(JOBS)} jobs to paginate")

## Pagination, three ways

| Style | Request | Response says | Notes |
|---|---|---|---|
| **Page number** | `?page=2&size=10` | `page`, `total` | simple; 🔴 breaks if items are added while you page |
| **Cursor / token** | `?cursor=abc` | `next_cursor` | 🔴 **stable under insertion** — the usual modern choice |
| **`Link` header** | `?page=2` | `Link: <...>; rel="next"` | RFC 8288; GitHub's style |

The naive loop looks like this — and it works, but it makes the caller think about pages.

In [ ]:
import requests

session = requests.Session()

# The obvious version: collect everything, page by page.
collected = []
page = 1
while True:
    response = session.get(f"{BASE}/jobs", params={"page": page, "size": 10}, timeout=5)
    response.raise_for_status()
    body = response.json()
    collected.extend(body["items"])
    if "next" not in response.links:          # requests parses the Link header for you
        break
    page += 1

print(f"collected {len(collected)} of {body['total']} jobs in {page} requests")
print("first:", collected[0], "| last:", collected[-1])
print()
print("response.links on a middle page:")
middle = session.get(f"{BASE}/jobs", params={"page": 1, "size": 10}, timeout=5)
print("   ", middle.links)

🔴 **`response.links` is the feature nobody knows about.** `requests` parses the
`Link` header into a dict, so `response.links["next"]["url"]` is the whole of "is there another
page?" — no string parsing.

## 🔴 Pagination belongs in a generator

The loop above forces every caller to know about pages, and it holds all 47 items in memory
before returning any of them. A **generator** (**4.3**) fixes both: the caller writes
`for job in fetch_jobs(...)` and pages appear as needed.

In [ ]:
from collections.abc import Iterator


from urllib.parse import urljoin


def iter_jobs(session, base, size=10) -> Iterator[dict]:
    """Yield every job, fetching pages lazily. The caller never sees a page."""
    url = f"{base}/jobs"
    params = {"page": 1, "size": size}
    while url:
        response = session.get(url, params=params, timeout=5)
        response.raise_for_status()
        yield from response.json()["items"]
        next_link = response.links.get("next")
        # 🔴 A Link header may be RELATIVE ("/jobs?page=2"). urljoin handles both.
        url = urljoin(response.url, next_link["url"]) if next_link else None
        params = None                       # the next URL already carries the query


before = len(REQUEST_LOG)
first_three = []
for job in iter_jobs(session, BASE):
    first_three.append(job)
    if len(first_three) == 3:
        break

print("took 3 items:", [j["id"] for j in first_three])
print(f"requests made: {len(REQUEST_LOG) - before}   🔴 ONE page, not five")

before = len(REQUEST_LOG)
everything = list(iter_jobs(session, BASE))
print(f"\ntook everything: {len(everything)} items"
      f" in {len(REQUEST_LOG) - before} requests")

done = [j for j in iter_jobs(session, BASE) if j["state"] == "done"]
print(f"filtered to done: {len(done)}")

**Three items cost one request.** The generator stopped as soon as the caller
did — which a list-returning function cannot do.

This is **4.3**'s lazy-evaluation argument applied to the network, where the saving is not
memory but *requests*.

### Cursor pagination

Cursor-based APIs hand you an opaque token instead of a page number. The generator shape is the
same, and it is **stable if items are inserted while you page** — a page-number paginator will
show you a duplicate or skip an item when that happens.

In [ ]:
def iter_jobs_by_cursor(session, base) -> Iterator[dict]:
    cursor = 0
    while cursor is not None:
        response = session.get(f"{base}/jobs/cursor",
                               params={"cursor": cursor}, timeout=5)
        response.raise_for_status()
        body = response.json()
        yield from body["items"]
        cursor = body["next_cursor"]            # None ends it


items = list(iter_jobs_by_cursor(session, BASE))
print(f"cursor pagination: {len(items)} items, last = {items[-1]['id']}")

## 🔴 The infinite pagination bug

The most common pagination failure is a loop that never ends, because the server always reports
a next page. Real APIs do this — through a bug, or because "next" means "there may be more".

**Always bound the loop.** Two guards, both cheap:

In [ ]:
def iter_broken(session, base, max_pages=None) -> Iterator[dict]:
    """The /jobs/broken endpoint ALWAYS reports a next_page."""
    page = 1
    seen_ids = set()
    while True:
        if max_pages is not None and page > max_pages:
            raise RuntimeError(f"pagination did not terminate after {max_pages} pages")
        response = session.get(f"{base}/jobs/broken", params={"page": page}, timeout=5)
        body = response.json()
        items = body["items"]

        if not items:                       # guard 1: an empty page means done
            return
        new = [j for j in items if j["id"] not in seen_ids]
        if not new:                         # guard 2: no NEW ids means done
            return
        seen_ids.update(j["id"] for j in new)
        yield from new
        page = body["next_page"]


print("--- with the empty-page and duplicate guards ---")
harvested = list(iter_broken(session, BASE))
print(f"   terminated cleanly with {len(harvested)} unique items")

print()
print("--- with no guards at all, bounded so this notebook can finish ---")


def iter_naive(session, base, stop_after=8):
    page = 1
    while True:
        if page > stop_after:
            print(f"   🔴 still going at page {page} - this loop never ends")
            return
        body = session.get(f"{base}/jobs/broken",
                           params={"page": page}, timeout=5).json()
        print(f"   page {page}: {len(body['items'])} items,"
              f" next_page={body['next_page']}")
        page = body["next_page"]


iter_naive(session, BASE)

The guarded version stopped at 47 items. The naive one was still asking for
page 9 of an empty collection when we cut it off — it would run forever, hammering the API.

🔴 **Three guards worth having in any paginator:**

1. **Stop on an empty page**, whatever the server claims about `next`.
2. **Stop when a page contains nothing new** — catches servers that loop.
3. **Cap the page count** and raise, so a bug is loud rather than silent.

## Rate limits

`429 Too Many Requests` means *slow down*. The response tells you how much:

| Header | Means |
|---|---|
| `Retry-After: 30` | 🔴 wait 30 seconds — **the server told you the answer** |
| `Retry-After: Wed, 21 Oct 2026 07:28:00 GMT` | the same, as a date |
| `X-RateLimit-Limit` | your quota per window |
| `X-RateLimit-Remaining` | 🔴 how many you have left — **throttle before you hit zero** |
| `X-RateLimit-Reset` | when the window resets (often a Unix timestamp) |

In [ ]:
import time
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime


def retry_after_seconds(value, default=1.0):
    """Retry-After arrives in two forms (the table above): seconds, or an HTTP date."""
    if value is None:
        return default
    try:
        return float(value)          # "Retry-After: 30"
    except ValueError:               # "Retry-After: Wed, 21 Oct 2026 07:28:00 GMT"
        when = parsedate_to_datetime(value)
        return max(0.0, (when - datetime.now(timezone.utc)).total_seconds())


def get_respecting_limits(session, url, max_attempts=5):
    """Honour Retry-After. Returns (response, attempts_made)."""
    for attempt in range(1, max_attempts + 1):
        response = session.get(url, timeout=5)
        remaining = response.headers.get("X-RateLimit-Remaining")

        if response.status_code != 429:
            return response, attempt, remaining

        wait = retry_after_seconds(response.headers.get("Retry-After"))
        print(f"   attempt {attempt}: 429, server said wait {wait}s"
              f" (remaining={remaining})")
        time.sleep(wait)

    raise RuntimeError("still rate limited after all attempts")


STATE["limited_calls"] = 0
response, attempts, remaining = get_respecting_limits(session, f"{BASE}/limited")
print(f"   attempt {attempts}: {response.status_code} {response.json()}"
      f" (remaining={remaining})")
print()
print("🔴 Retry-After is not advice - it is the server telling you exactly")
print("   how long to wait. Ignoring it and retrying immediately is how a")
print("   client gets its API key suspended.")

## Backoff, and 🔴 why it needs jitter

When there is no `Retry-After`, you have to choose the wait yourself. **15.1** built exactly
this function — `retry_delay(attempt)` with exponential growth and a ceiling — as its first
example. Here is why it needs one more ingredient.

```
   fixed        1s  1s  1s  1s  1s     hammers a struggling server
   exponential  1s  2s  4s  8s  16s    much better
   jittered     0.7 1.6 3.1 7.2 12.4   🔴 and clients stop synchronising
```

🔴 **The thundering herd:** if a server goes down, every client retries at the same moment. With
pure exponential backoff they *stay* synchronised — all of them hit at 1s, then all at 2s, then
all at 4s — and each wave knocks the server over again. **Jitter spreads them out.**

In [ ]:
import random


def delay_fixed(attempt, base=1.0, ceiling=30.0):
    return base


def delay_exponential(attempt, base=1.0, ceiling=30.0):
    return min(base * 2 ** attempt, ceiling)


def delay_jittered(attempt, base=1.0, ceiling=30.0, rng=random):
    """Full jitter: a random point in [0, exponential]. AWS's recommendation."""
    return rng.uniform(0, min(base * 2 ** attempt, ceiling))


rng = random.Random(183)          # seeded so this cell is reproducible (15.9)

print(f"{'attempt':>8}{'fixed':>9}{'exponential':>14}{'jittered':>11}")
print("-" * 44)
for attempt in range(6):
    print(f"{attempt:>8}{delay_fixed(attempt):>9.2f}"
          f"{delay_exponential(attempt):>14.2f}"
          f"{delay_jittered(attempt, rng=rng):>11.2f}")

print()
print("--- the thundering herd, simulated: 200 clients, attempt 3 ---")
for label, delay in (("exponential", delay_exponential), ("jittered", delay_jittered)):
    waits = [delay(3, rng=random.Random(seed)) if label == "jittered"
             else delay(3) for seed in range(200)]
    buckets = {}
    for wait in waits:
        buckets[round(wait)] = buckets.get(round(wait), 0) + 1
    busiest = max(buckets.values())
    print(f"   {label:12} {len(buckets):2} distinct wait times,"
          f" busiest second holds {busiest:3} of 200 clients")

With pure exponential backoff **all 200 clients retry in the same second**.
With jitter they spread across the window, and the server sees a trickle instead of a wall.

> **Variants you will see named:** *full jitter* (above, `uniform(0, backoff)`), *equal jitter*
> (`backoff/2 + uniform(0, backoff/2)`), and *decorrelated jitter*. Full jitter is the usual
> recommendation and the simplest to reason about.

## 🔴 What to retry — and what never to

Retrying the wrong thing is worse than not retrying. Two questions decide it: **did it fail in
a way that might succeed later**, and **is repeating it safe** (**18.1**'s idempotency table)?

In [ ]:
from requests import exceptions as rex

RETRYABLE_STATUS = {408, 425, 429, 500, 502, 503, 504}
IDEMPOTENT_METHODS = {"GET", "HEAD", "OPTIONS", "PUT", "DELETE"}


def should_retry(method, status=None, error=None, has_idempotency_key=False):
    """The whole policy, in one readable function."""
    if error is not None:
        # A connection error or timeout: the request may never have arrived.
        if method in IDEMPOTENT_METHODS or has_idempotency_key:
            return True, "transport failure, safe to repeat"
        return False, "transport failure, but repeating may duplicate the write"
    if status in RETRYABLE_STATUS:
        if method in IDEMPOTENT_METHODS or has_idempotency_key:
            return True, f"{status} is transient"
        return False, f"{status} is transient, but repeating may duplicate the write"
    if status is not None and 400 <= status < 500:
        return False, f"{status} is your mistake - retrying cannot fix it"
    return False, "succeeded, or nothing to retry"


cases = [
    ("GET", 503, None, False),
    ("GET", 429, None, False),
    ("GET", 404, None, False),
    ("GET", 403, None, False),
    ("POST", 503, None, False),
    ("POST", 503, None, True),
    ("PUT", 500, None, False),
    ("GET", None, rex.ConnectTimeout(), False),
    ("POST", None, rex.ConnectTimeout(), False),
    ("POST", None, rex.ConnectTimeout(), True),
]

print(f"{'method':8}{'status':>8}{'idem-key':>10}  {'retry?':8} why")
print("-" * 86)
for method, status, error, key in cases:
    retry, why = should_retry(method, status, error, key)
    shown = str(status) if status else type(error).__name__
    print(f"{method:8}{shown:>8}{str(key):>10}  {str(retry):8} {why}")

Read the two `POST` + `503` rows. **The only difference is an idempotency
key**, and it flips the decision — which is the whole reason the next section exists.

🔴 **Never blindly retry `400`, `401`, `403`, `404`, `422`.** They are statements about *your*
request, and repeating it unchanged cannot fix them — it wastes quota and, with `401`, can
lock an account. The one earned exception: `401` is worth retrying **exactly once, after
refreshing credentials** — a *changed* request, which is precisely what **18.2**'s
`RefreshingClient` does.

## Retries for free: `urllib3.Retry`

You rarely need to write the loop. `requests` sits on `urllib3`, which has a battle-tested
retry policy — mount it on an `HTTPAdapter` and every request through that session gets it.

In [ ]:
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

policy = Retry(
    total=5,
    backoff_factor=0.1,                     # sleeps 0, 0.2, 0.4, 0.8 ... - first retry is immediate
    backoff_jitter=0.1,                     # 🔴 urllib3 2.x adds jitter for you
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods={"GET", "HEAD", "PUT", "DELETE", "OPTIONS"},   # NOT POST
    respect_retry_after_header=True,        # honour the server's instruction
    raise_on_status=False,
)

resilient = requests.Session()
resilient.mount("http://", HTTPAdapter(max_retries=policy))
resilient.mount("https://", HTTPAdapter(max_retries=policy))

STATE["flaky_calls"] = 0
before = len(REQUEST_LOG)
started = time.perf_counter()
response = resilient.get(f"{BASE}/flaky", timeout=5)
elapsed = (time.perf_counter() - started) * 1000

print(f"/flaky fails twice out of every three calls.")
print(f"   final status : {response.status_code} {response.json()}")
print(f"   HTTP requests actually sent: {len(REQUEST_LOG) - before}")
print(f"   wall time    : {elapsed:.0f} ms")
print()
print("   🔴 The caller saw ONE call. urllib3 made three.")

print()
print("--- and it does NOT retry what it should not ---")
before = len(REQUEST_LOG)
forbidden = resilient.get(f"{BASE}/forbidden", timeout=5)
print(f"   403 -> {len(REQUEST_LOG) - before} request(s), status {forbidden.status_code}")
print("   403 is not in status_forcelist, so it was returned immediately.")

Three HTTP requests, one line of caller code, and a `403` returned instantly
rather than retried.

🔴 **`allowed_methods` excludes `POST` by default, and that default is correct.** Overriding it
without an idempotency key is how you create duplicate orders.

## Idempotency keys — making `POST` safe

A `POST` that times out leaves you in the worst position: **you do not know whether it
happened.** Retrying might create a second record; not retrying might lose the first.

An **idempotency key** solves it. You generate a unique key per logical operation and send it
with the request. The server records the key with the result, and if it sees the same key again
it **returns the original result instead of doing the work twice**.

In [ ]:
import uuid

key = f"job-create-{uuid.uuid4()}"
payload = {"region": "eu", "state": "queued"}

first = session.post(f"{BASE}/jobs", json=payload, timeout=5,
                     headers={"Idempotency-Key": key})
print(f"first  send: {first.status_code} {first.json()}"
      f"  replayed={first.headers.get('X-Replayed', 'no')}")

# Exactly what a retry after a timeout would do: same key, same body.
second = session.post(f"{BASE}/jobs", json=payload, timeout=5,
                      headers={"Idempotency-Key": key})
print(f"retry  send: {second.status_code} {second.json()}"
      f"  replayed={second.headers.get('X-Replayed', 'no')}")

print(f"\n   same id returned : {first.json()['id'] == second.json()['id']}")
print(f"   records created  : {len(STATE['created'])}")

# Without a key, the same retry creates a second record.
no_key_1 = session.post(f"{BASE}/jobs", json=payload, timeout=5)
no_key_2 = session.post(f"{BASE}/jobs", json=payload, timeout=5)
print(f"\n   without a key: {no_key_1.json()['id']} and {no_key_2.json()['id']}"
      f"  🔴 two different records from one intent")

With a key the retry returned the **original** record and `X-Replayed: true`.
Without one, the same two calls created **two different jobs**.

🔴 **Generate the key from the *operation*, not the attempt.** A fresh `uuid4()` on every retry
defeats the entire mechanism — the key must be stable across retries of the same logical action.

> Stripe, Square and most payment APIs require this header. If an API supports it, use it for
> every non-idempotent request; it is what makes `POST` retryable at all.

## Circuit breakers, briefly

Retries help with a *blip*. When a dependency is properly down, retrying makes things worse for
everyone — you add load to a service that is already failing, and your own threads pile up
waiting.

A **circuit breaker** wraps the client and tracks failures:

```
   CLOSED ──(N failures in a row)──> OPEN ──(after a cooldown)──> HALF-OPEN
      ▲                               │                              │
      │                          fail fast,                    let ONE through
      └────────(it succeeded)─────────┴──────────────────────────────┘
```

- **Closed** — normal; requests flow.
- **Open** — 🔴 fail *immediately* without calling the API at all, for a cooldown period.
- **Half-open** — let a single request through; promote to closed on success, back to open on
  failure.

The practical value is **failing fast**: when the API is down, your service returns an error in
microseconds instead of tying up every worker for a 30-second timeout. Libraries: `pybreaker`,
or `tenacity` for the retry half.

In [ ]:
# ---- tidy up ----
SERVER.shutdown()
print("fake API stopped")
print("HTTP requests it handled:", len(REQUEST_LOG))

---

## Common Mistakes & Pitfalls

1. 🔴 **A pagination loop with no exit guard.** Trust an empty page and a lack of new IDs, not the server's `next` flag alone — and cap the page count.
2. **Returning a list from a paginator instead of a generator.** The caller cannot stop early, and you fetch every page to answer a question about the first (**4.3**).
3. **Parsing the `Link` header by hand.** `requests` gives you `response.links`.
4. 🔴 **Ignoring `Retry-After`.** The server told you exactly how long to wait; guessing shorter is how a key gets suspended.
5. 🔴 **Exponential backoff with no jitter.** Every client retries in the same second and the recovering server falls over again.
6. **Retrying `4xx`.** `400`, `403`, `404` and `422` are statements about your request.
7. 🔴 **Retrying a `POST` without an idempotency key.** You may create the thing twice, and a timeout means you cannot tell.
8. **Generating a new idempotency key per attempt.** The key must identify the *operation*, not the try.
9. **Retrying forever.** Bound the attempts and surface the failure; an infinite retry is an outage you cannot see.
10. **Using page-number pagination on data that changes.** Items shift between pages; use a cursor.

## Best Practices

- Expose pagination as a generator yielding items, never pages.
- Use `response.links` for `Link` headers and prefer cursor pagination when offered.
- Guard every paginator: empty page, no new IDs, and a maximum page count.
- Honour `Retry-After` exactly; watch `X-RateLimit-Remaining` and slow down *before* zero.
- Use exponential backoff **with full jitter**, and a ceiling.
- Let `urllib3.Retry` on an `HTTPAdapter` do the work; keep `POST` out of `allowed_methods` unless you send an idempotency key.
- Send an idempotency key with every non-idempotent request the API supports it for.
- Bound total retry time, not just attempt count — a caller is waiting.
- Add a circuit breaker when a dependency being down should fail fast rather than queue.
- Log every retry with the attempt number and reason (**15.10**) — silent retries hide a degrading dependency.

## Practice Exercises

Try these before moving on.

1. Rewrite `iter_jobs` to accept a `limit` and stop after that many items. How many requests does `limit=5` cost?
2. 🔴 Remove the guards from `iter_broken` and add a hard page cap instead. Which failure would you rather debug at 3 a.m.?
3. Point the cursor paginator at `/jobs` and the page paginator at `/jobs/cursor`. What breaks, and what does that tell you about coupling to a pagination style?
4. Write a client that throttles itself when `X-RateLimit-Remaining` drops below 20% of the limit, instead of waiting for a `429`.
5. 🔴 Simulate 500 clients with fixed, exponential and jittered backoff. Plot how many retry in each second. Which one recovers the server fastest?
6. Configure `Retry` to include `POST` and prove, using the fake API's `/jobs` endpoint without an idempotency key, that a flaky connection creates duplicates.
7. Implement a circuit breaker with the three states. Make it open after 3 failures and half-open after 2 seconds.
8. Extend `should_retry` to take an elapsed-time budget, so a slow sequence of retries gives up even when attempts remain.
9. **Interview question:** a `POST` times out. Did it succeed? What do you do, and what should the API have given you to make the answer knowable?

---

## Version notes

| Version | Change |
|---|---|
| **urllib3 2.x** | 🔴 `backoff_jitter` added to `Retry`; `allowed_methods` replaced the old `method_whitelist` |
| **urllib3 1.x** | honouring `Retry-After` is *old* default behaviour — automatic since 1.19 (2016), with the `respect_retry_after_header` opt-out since 1.25. None of it is new in 2.x |
| **requests 2.x** | `response.links` parses RFC 8288 `Link` headers |
| **Python 3.12** | `itertools.batched` — useful when you must send *pages* rather than receive them |

> **Libraries worth knowing.** `tenacity` gives declarative retries (`@retry(wait=wait_random_exponential(...))`)
> with far more control than `urllib3.Retry`; `pybreaker` implements the circuit breaker;
> `httpx` (**18.5**) has its own transport-level retry configuration.

## Where next

| Notebook | Covers |
|---|---|
| **18.4** | validating what you receive, with `pydantic` |
| **18.5** | testing API clients, and concurrency for I/O-bound work |

## Related

- **18.1** — the idempotency table this notebook's retry policy is built on
- **18.2** — `401` and token refresh: the one `4xx` worth retrying once, *after* refreshing credentials — never blindly
- **4.3 Generators** — laziness, which here saves requests rather than memory
- **15.1** — `retry_delay()`, the very first function in the testing folder
- **15.9** — seeding the RNG so a jitter demonstration is reproducible
- **15.10 Logging** — where retry attempts should be recorded